# ANALIZA DANYCH Z ZEGARKA APPLE WATCH


## Podstawowe informacje o pliku:

* Plik jest pobierany z aplikacji Zdrowie (Apple Health) na telefonie
* format: XML
* struktura: hierarchiczna, z wieloma różnymi typami rekordów (Record, Correlation, Workout, ActivitySummary, ClinicalRecord, Audiogram, VisionPrescription)
* zawiera dane dotyczące zdrowia, aktywności, snu, tętna, kroków, itp.
* jest dość duży ponad 700 mb





## TO DO:
1. [x] rozgryźć strukturę pliku xml - pól w nim zawartych - co dokładnie w nim jest
2. [x] sprawdzić pomiar EKG i możliwość ściągnięcia danych
3. [x] napisać kod pythona do wczytania tych danych
4. [x] zastanowić się nad programem ćwiczeń dla osób o różnej kondycji, z pomiarem zegarkiem

In [150]:
import os
import streamlit as st
import time

from altair.datasets import data
from lxml import etree as ET
import pandas as pd
from parser import load_one_type, load_all_health_data


In [151]:
path = "eksport.xml"

In [152]:
def discover_xml_structure(file_path):
    # Słownik: Klucz = Nazwa Tagu, Wartość = Zbiór atrybutów 'type' lub 'workoutActivityType'
    found_structures = {}

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        tag_name = elem.tag

        #  dodajemy do slownika jesli nie ma w nim jeszcze
        if tag_name not in found_structures:
            found_structures[tag_name] = set()

        # sprawdzamy typy głównych tagów
        sub_type = elem.get("type") or elem.get("workoutActivityType") or "Brak"

        found_structures[tag_name].add(sub_type)

        # Czyścimy RAM
        elem.clear()

    return found_structures

moje_dane = discover_xml_structure(path)

print("ODKRYTE TAGI + typy danych")
for tag, sub_types in moje_dane.items():
    print(f"\n TAG: <{tag}>")
    print(f"   Liczba różnych typów: {len(sub_types)}")
    for st in list(sub_types):
        print(f"     - {st}")

ODKRYTE TAGI + typy danych

 TAG: <ExportDate>
   Liczba różnych typów: 1
     - Brak

 TAG: <Me>
   Liczba różnych typów: 1
     - Brak

 TAG: <Record>
   Liczba różnych typów: 41
     - HKQuantityTypeIdentifierStairDescentSpeed
     - HKQuantityTypeIdentifierEnvironmentalSoundReduction
     - HKQuantityTypeIdentifierWalkingHeartRateAverage
     - HKQuantityTypeIdentifierHeadphoneAudioExposure
     - HKCategoryTypeIdentifierMenstrualFlow
     - HKDataTypeSleepDurationGoal
     - HKQuantityTypeIdentifierHeartRate
     - HKCategoryTypeIdentifierAudioExposureEvent
     - HKQuantityTypeIdentifierVO2Max
     - HKQuantityTypeIdentifierBodyMass
     - HKQuantityTypeIdentifierBodyFatPercentage
     - HKQuantityTypeIdentifierBasalEnergyBurned
     - HKQuantityTypeIdentifierDistanceCycling
     - HKQuantityTypeIdentifierAppleExerciseTime
     - HKQuantityTypeIdentifierWalkingStepLength
     - HKQuantityTypeIdentifierAppleStandTime
     - HKQuantityTypeIdentifierHeartRateVariabilitySDNN
     - H

In [153]:
important_tags = {
    'Me',"Record", "Workout", "ActivitySummary", "InstantaneousBeatsPerMinute"
}


In [154]:
print(moje_dane)

{'ExportDate': {'Brak'}, 'Me': {'Brak'}, 'Record': {'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKCategoryTypeIdentifierMenstrualFlow', 'HKDataTypeSleepDurationGoal', 'HKQuantityTypeIdentifierHeartRate', 'HKCategoryTypeIdentifierAudioExposureEvent', 'HKQuantityTypeIdentifierVO2Max', 'HKQuantityTypeIdentifierBodyMass', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierBasalEnergyBurned', 'HKQuantityTypeIdentifierDistanceCycling', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierWalkingStepLength', 'HKQuantityTypeIdentifierAppleStandTime', 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'HKQuantityTypeIdentifierTimeInDaylight', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKQuantityTypeIdentifierWalkingSpeed', 'HKQuantityTypeIdentifierWalkingAsymme

In [155]:

diff_types = set()

diff_tags = set()
# Iterujemy po pliku
for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag in important_tags:
        typ = elem.get("type")  or elem.get("workoutActivityType") or elem.tag

        if typ not in diff_types:
            print(f"\n TYP: {typ}")
            print(f"   Atrybuty: {elem.attrib}")


            children = list(elem)
            if children:
                for child in children:
                    print(f"   typ: <{child.tag}> atrybuty: {child.attrib}")
            diff_types.add(typ)



        elem.clear()




 TYP: Me
   Atrybuty: {'HKCharacteristicTypeIdentifierDateOfBirth': '2006-03-02', 'HKCharacteristicTypeIdentifierBiologicalSex': 'HKBiologicalSexFemale', 'HKCharacteristicTypeIdentifierBloodType': 'HKBloodTypeNotSet', 'HKCharacteristicTypeIdentifierFitzpatrickSkinType': 'HKFitzpatrickSkinTypeNotSet', 'HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse': 'Brak'}

 TYP: HKQuantityTypeIdentifierHeight
   Atrybuty: {'type': 'HKQuantityTypeIdentifierHeight', 'sourceName': 'FitnessOnline', 'sourceVersion': '206', 'unit': 'cm', 'creationDate': '2025-08-12 11:58:06 +0100', 'startDate': '2025-08-12 11:58:05 +0100', 'endDate': '2025-08-12 11:58:05 +0100', 'value': '163'}

 TYP: HKQuantityTypeIdentifierBodyMass
   Atrybuty: {'type': 'HKQuantityTypeIdentifierBodyMass', 'sourceName': 'Zdrowie', 'sourceVersion': '14.7.1', 'unit': 'kg', 'creationDate': '2021-09-06 10:46:43 +0100', 'startDate': '2021-09-06 10:46:00 +0100', 'endDate': '2021-09-06 10:46:00 +0100', 'value': '55'}
   typ: <Metadat

In [156]:
print(diff_types)
print(len(diff_types))

{'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKCategoryTypeIdentifierMenstrualFlow', 'HKDataTypeSleepDurationGoal', 'HKQuantityTypeIdentifierHeartRate', 'HKCategoryTypeIdentifierAudioExposureEvent', 'HKQuantityTypeIdentifierVO2Max', 'HKQuantityTypeIdentifierBodyMass', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierBasalEnergyBurned', 'InstantaneousBeatsPerMinute', 'HKQuantityTypeIdentifierDistanceCycling', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierWalkingStepLength', 'HKQuantityTypeIdentifierAppleStandTime', 'Me', 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'HKWorkoutActivityTypeCycling', 'HKQuantityTypeIdentifierTimeInDaylight', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKWorkoutActivityTypePilates', 'HKQuantityTypeIdentifierWal

In [157]:
#Stworzenie dataframe z wybranym typem danych

short_names = {
    # --- CIAŁO / PODSTAWOWE ---
    'HKQuantityTypeIdentifierHeight': 'height',
    'HKQuantityTypeIdentifierBodyMass': 'weight',
    'HKQuantityTypeIdentifierWaistCircumference': 'waist',
    'HKQuantityTypeIdentifierBodyFatPercentage': 'fat_pct',

    # --- SERCE (HEART) ---
    'HKQuantityTypeIdentifierHeartRate': 'hr',
    'HKQuantityTypeIdentifierRestingHeartRate': 'rhr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage': 'hr_walk_avg',
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute': 'hrr_1min',
    'HKCategoryTypeIdentifierHighHeartRateEvent': 'hr_high_event',

    # --- AKTYWNOŚĆ / ENERGIA ---
    'HKQuantityTypeIdentifierStepCount': 'steps',
    'HKQuantityTypeIdentifierDistanceWalkingRunning': 'dist_walk',
    'HKQuantityTypeIdentifierDistanceCycling': 'dist_cycle',
    'HKQuantityTypeIdentifierBasalEnergyBurned': 'kcal_basal',
    'HKQuantityTypeIdentifierActiveEnergyBurned': 'kcal_active',
    'HKQuantityTypeIdentifierFlightsClimbed': 'flights',
    'HKQuantityTypeIdentifierAppleExerciseTime': 'exercise_min',
    'HKQuantityTypeIdentifierAppleStandTime': 'stand_min',
    'HKCategoryTypeIdentifierAppleStandHour': 'stand_hr',
    'HKQuantityTypeIdentifierPhysicalEffort': 'effort',
    'HKQuantityTypeIdentifierVO2Max': 'vo2max',

    # --- MOBILNOŚĆ / CHÓD ---
    'HKQuantityTypeIdentifierWalkingSpeed': 'walk_speed',
    'HKQuantityTypeIdentifierWalkingStepLength': 'walk_step_len',
    'HKQuantityTypeIdentifierWalkingAsymmetryPercentage': 'walk_asym',
    'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage': 'walk_dbl_supp',
    'HKQuantityTypeIdentifierStairAscentSpeed': 'stair_up_speed',
    'HKQuantityTypeIdentifierStairDescentSpeed': 'stair_down_speed',
    'HKQuantityTypeIdentifierAppleWalkingSteadiness': 'walk_steadiness',
    'HKQuantityTypeIdentifierSixMinuteWalkTestDistance': 'walk_6min',

    # --- ODDECH / POZIOMY ---
    'HKQuantityTypeIdentifierOxygenSaturation': 'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate': 'resp_rate',

    # --- SEN ---
    'HKCategoryTypeIdentifierSleepAnalysis': 'sleep',
    'HKDataTypeSleepDurationGoal': 'sleep_goal',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',

    # --- ŚRODOWISKO / DŹWIĘK ---
    'HKQuantityTypeIdentifierEnvironmentalAudioExposure': 'audio_env',
    'HKQuantityTypeIdentifierHeadphoneAudioExposure': 'audio_hp',
    'HKQuantityTypeIdentifierEnvironmentalSoundReduction': 'sound_red',
    'HKCategoryTypeIdentifierAudioExposureEvent': 'audio_event',
    'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent': 'audio_hp_event',
    'HKQuantityTypeIdentifierTimeInDaylight': 'daylight',

    # --- INNE ---
    'HKCategoryTypeIdentifierMenstrualFlow': 'menstr'
}


In [158]:
def load_one_type(file_path, tag_name, type_name=None):
    """
    Pobiera dane na podstawie samego tagu (np. 'ActivitySummary') lub tagu i typu (np. 'Record' + 'hr').
    Chirurgiczna pęseta do eksploracji danych.
    """
    health_data = []

    if file_path is None:
        return "Zła ścieżka pliku"

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:

        if elem.tag == tag_name:
            current_type = elem.get('type') or elem.get('workoutActivityType')
            if type_name is None or current_type == type_name:
                record = dict(elem.attrib)

                for child in elem:
                    if child.tag == "MetadataEntry":
                        key = child.get('key')
                        if key is not None:
                            record[key] = child.get('value')
                health_data.append(record)

            elem.clear()

    if health_data:
        df = pd.DataFrame(health_data)
        if 'value' in df.columns:
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
        for date_col in ['startDate', 'endDate', 'creationDate']:
            if date_col in df.columns:
                df[date_col] = pd.to_datetime(df[date_col])
        return df
    return pd.DataFrame()


df_hr = load_one_type(path, 'Record', 'HKQuantityTypeIdentifierHeartRate')
display(df_hr.head())


,type,sourceName,sourceVersion,unit,creationDate,startDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext,HKMetadataKeySyncVersion,HKMetadataKeySyncIdentifier
0,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 16:29:14+01:00,2021-12-17 00:00:00+01:00,2021-12-17 00:00:59+01:00,72.0,NaN,NaN,NaN,NaN
1,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 16:29:14+01:00,2021-12-17 00:01:00+01:00,2021-12-17 00:01:59+01:00,70.0,NaN,NaN,NaN,NaN
2,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 16:29:14+01:00,2021-12-17 00:02:00+01:00,2021-12-17 00:02:59+01:00,70.0,NaN,NaN,NaN,NaN
3,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 16:29:14+01:00,2021-12-17 00:03:00+01:00,2021-12-17 00:03:59+01:00,69.0,NaN,NaN,NaN,NaN
4,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 16:29:14+01:00,2021-12-17 00:04:00+01:00,2021-12-17 00:04:59+01:00,70.0,NaN,NaN,NaN,NaN


Analiza samych danych tetna


In [159]:
print(df_hr.columns)
print(df_hr.head())
print(df_hr.describe())


Index(['type', 'sourceName', 'sourceVersion', 'unit', 'creationDate',
       'startDate', 'endDate', 'value', 'device',
       'HKMetadataKeyHeartRateMotionContext', 'HKMetadataKeySyncVersion',
       'HKMetadataKeySyncIdentifier'],
      dtype='object')
                                type sourceName sourceVersion       unit  \
0  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
1  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
2  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
3  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
4  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   

               creationDate                 startDate  \
0 2021-12-24 16:29:14+01:00 2021-12-17 00:00:00+01:00   
1 2021-12-24 16:29:14+01:00 2021-12-17 00:01:00+01:00   
2 2021-12-24 16:29:14+01:00 2021-12-17 00:02:00+01:00   
3 2021-12-24 16:29:14+01:00 2021-12-17 00:03:00+01:00   
4 2

Czyścimy  dane - usuwamy kolumny w których wszystkie wartości sa nan

In [160]:
df_hr = df_hr.dropna(axis=1, how='all')

In [161]:
# Usuwamy konkretne kolumny techniczne, których nie potrzebujemy do analizy tętna
df_hr = df_hr.drop(columns=['HKMetadataKeySyncVersion', 'HKMetadataKeySyncIdentifier'], errors='ignore')

In [162]:
print(df_hr.columns)

Index(['type', 'sourceName', 'sourceVersion', 'unit', 'creationDate',
       'startDate', 'endDate', 'value', 'device',
       'HKMetadataKeyHeartRateMotionContext'],
      dtype='object')


In [163]:
# Usuwamy rekordy, które mają IDENTYCZNY start, koniec I wartość
df_hr = df_hr.drop_duplicates(subset=['startDate', 'endDate', 'value'])

print(f"Usunięto {len(df_hr) - len(df_hr)} idealnych duplikatów.")

Usunięto 0 idealnych duplikatów.


In [164]:

# bierzemy dane pobierane tylko z zegarka
df = df_hr[df_hr['sourceName'].str.contains('Apple', na=False)]


# Wywalamy rekordy, gdzie koniec jest przed początkiem i jakieś błędne
df = df[df['endDate'] >= df['startDate']]

# indeksujemy po dacie poczatkowej
df = df.set_index('startDate')




In [165]:
display(df.tail(100))

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2026-03-05 14:31:23+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:31:26+01:00,2026-03-05 14:31:23+01:00,140.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:31:31+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:31:31+01:00,2026-03-05 14:31:31+01:00,138.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:31:33+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:31:37+01:00,2026-03-05 14:31:33+01:00,138.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:31:37+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:31:42+01:00,2026-03-05 14:31:37+01:00,139.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:31:45+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:31:47+01:00,2026-03-05 14:31:45+01:00,139.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
...,...,...,...,...,...,...,...,...,...
2026-03-05 14:39:22+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:39:27+01:00,2026-03-05 14:39:22+01:00,107.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:39:27+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:39:32+01:00,2026-03-05 14:39:27+01:00,107.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:39:32+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:39:37+01:00,2026-03-05 14:39:32+01:00,101.0,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2


In [167]:

# 2. Definiujemy reguły: dla 'value' średnia, dla reszty 'pierwszy element'
# Tworzymy słownik, który dla każdej kolumny przypisuje funkcję 'first'
agg_rules = {col: 'first' for col in df.columns}

# Nadpisujemy regułę dla kolumny 'value' na 'mean'
if 'value' in agg_rules:
    agg_rules['value'] = 'mean'

# 3. Magiczne resamplowanie z regułami
df_minutowe = df.resample('1min').agg(agg_rules)

# 4. Przywracamy indeks do kolumny
df_minutowe = df_minutowe.reset_index()

display(df_minutowe)

,startDate,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
0,2024-03-03 16:10:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:10:31+01:00,2024-03-03 16:10:22+01:00,93.0,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
1,2024-03-03 16:11:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
2,2024-03-03 16:12:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:16:07+01:00,2024-03-03 16:12:16+01:00,84.0,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
3,2024-03-03 16:13:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
4,2024-03-03 16:14:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...
1054006,2026-03-05 14:56:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
1054007,2026-03-05 14:57:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
1054008,2026-03-05 14:58:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None
1054009,2026-03-05 14:59:00+01:00,None,None,None,None,NaT,NaT,NaN,None,None


In [168]:
df = df_minutowe.dropna()
display(df)

,startDate,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
0,2024-03-03 16:10:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:10:31+01:00,2024-03-03 16:10:22+01:00,93.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
2,2024-03-03 16:12:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:16:07+01:00,2024-03-03 16:12:16+01:00,84.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
8,2024-03-03 16:18:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:21:08+01:00,2024-03-03 16:18:43+01:00,82.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
14,2024-03-03 16:24:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:25:36+01:00,2024-03-03 16:24:23+01:00,80.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
15,2024-03-03 16:25:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:30:49+01:00,2024-03-03 16:25:28+01:00,78.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
...,...,...,...,...,...,...,...,...,...,...
1053986,2026-03-05 14:36:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:36:02+01:00,2026-03-05 14:36:01+01:00,112.615385,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
1053987,2026-03-05 14:37:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:37:07+01:00,2026-03-05 14:37:03+01:00,102.818182,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
1053988,2026-03-05 14:38:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:38:07+01:00,2026-03-05 14:38:02+01:00,90.272727,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
1053989,2026-03-05 14:39:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:39:02+01:00,2026-03-05 14:39:01+01:00,97.777778,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2


In [169]:

df.set_index('startDate', inplace=True)
df.sort_index(inplace=True)

In [170]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 168353 entries, 2024-03-03 16:10:00+01:00 to 2026-03-05 15:00:00+01:00
Data columns (total 9 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   type                                 168353 non-null  object                   
 1   sourceName                           168353 non-null  object                   
 2   sourceVersion                        168353 non-null  object                   
 3   unit                                 168353 non-null  object                   
 4   creationDate                         168353 non-null  datetime64[ns, UTC+01:00]
 5   endDate                              168353 non-null  datetime64[ns, UTC+01:00]
 6   value                                168353 non-null  float64                  
 7   device                               168353 non-null  object              

In [171]:
df = df.dropna(subset=['value'])
print(df.info())
df

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 168353 entries, 2024-03-03 16:10:00+01:00 to 2026-03-05 15:00:00+01:00
Data columns (total 9 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   type                                 168353 non-null  object                   
 1   sourceName                           168353 non-null  object                   
 2   sourceVersion                        168353 non-null  object                   
 3   unit                                 168353 non-null  object                   
 4   creationDate                         168353 non-null  datetime64[ns, UTC+01:00]
 5   endDate                              168353 non-null  datetime64[ns, UTC+01:00]
 6   value                                168353 non-null  float64                  
 7   device                               168353 non-null  object              

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2024-03-03 16:10:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:10:31+01:00,2024-03-03 16:10:22+01:00,93.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
2024-03-03 16:12:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:16:07+01:00,2024-03-03 16:12:16+01:00,84.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
2024-03-03 16:18:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:21:08+01:00,2024-03-03 16:18:43+01:00,82.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
2024-03-03 16:24:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:25:36+01:00,2024-03-03 16:24:23+01:00,80.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
2024-03-03 16:25:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 16:30:49+01:00,2024-03-03 16:25:28+01:00,78.000000,"<<HKDevice: 0x5c4ace260>, name:Apple Watch, ma...",1
...,...,...,...,...,...,...,...,...,...
2026-03-05 14:36:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:36:02+01:00,2026-03-05 14:36:01+01:00,112.615385,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:37:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:37:07+01:00,2026-03-05 14:37:03+01:00,102.818182,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:38:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:38:07+01:00,2026-03-05 14:38:02+01:00,90.272727,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2


In [172]:
df.loc['2026-3-3':]

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2026-03-03 00:05:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:09:08+01:00,2026-03-03 00:05:13+01:00,84.000000,"<<HKDevice: 0x5c4acc7d0>, name:Apple Watch, ma...",0
2026-03-03 00:13:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:15:07+01:00,2026-03-03 00:13:34+01:00,79.000000,"<<HKDevice: 0x5c4acc7d0>, name:Apple Watch, ma...",0
2026-03-03 00:14:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:19:31+01:00,2026-03-03 00:14:50+01:00,81.000000,"<<HKDevice: 0x5c4acc7d0>, name:Apple Watch, ma...",0
2026-03-03 00:19:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:19:24+01:00,2026-03-03 00:19:24+01:00,97.000000,"<<HKDevice: 0x5c4acc7d0>, name:Apple Watch, ma...",0
2026-03-03 00:23:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:24:34+01:00,2026-03-03 00:23:10+01:00,84.000000,"<<HKDevice: 0x5c4acc7d0>, name:Apple Watch, ma...",0
...,...,...,...,...,...,...,...,...,...
2026-03-05 14:36:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:36:02+01:00,2026-03-05 14:36:01+01:00,112.615385,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:37:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:37:07+01:00,2026-03-05 14:37:03+01:00,102.818182,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2
2026-03-05 14:38:00+01:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-05 14:38:07+01:00,2026-03-05 14:38:02+01:00,90.272727,"<<HKDevice: 0x5c4acfe30>, name:Apple Watch, ma...",2


In [ ]:
df.loc['2024-03-01':'2024-03-05']

In [ ]:
%%sql
